Complete User Data Export from EnergyID

This notebook connects to the EnergyID API to perform a full export of all data associated with the authenticated user. The process is broken down into several steps:

Setup & Configuration: Load API credentials and prepare the environment.

Fetch All Dossiers (Records): Retrieve a list of all dossiers for the user.

Fetch All Meters: Iterate through each dossier to get a comprehensive list of all meters.

Fetch All Readings: Iterate through every meter to download all its time-series data, handling pagination where necessary. This is the most time-consuming step.

Archive the Data: Zip the entire output directory for easy sharing with a data analyst.

The final output will be a collection of CSV files organized in a timestamped directory, which is then compressed into a single .zip file.

In [ ]:
import polars as pl
import requests
import os
import time
import json
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
import zipfile

print("Libraries imported successfully.")

# Load environment variables from a .env file
load_dotenv()

# --- Configuration ---
API_KEY = os.getenv("API_KEY")
BASE_API_URL = os.getenv("BASE_API_URL", "https://api.energyid.eu")

# --- Prepare Headers for API Requests ---
headers = {
    "Authorization": f"apiKey {API_KEY}",
    "Content-Type": "application/json",
    "Accept": "application/json",
}

# --- Create a unique directory for this export ---
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPORT_DIR = Path(f"energyid_export_{timestamp}")
READINGS_DIR = EXPORT_DIR / "readings"

try:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    READINGS_DIR.mkdir(exist_ok=True)
    print(f"Export directory created at: '{EXPORT_DIR}'")
except Exception as e:
    print(f"Error creating export directory: {e}")

In [ ]:
# A quick check to confirm the API key and base URL are working.
try:
    print("--- Testing API Connection ---")
    response = requests.get(f"{BASE_API_URL}/api/v1/members/me", headers=headers)
    response.raise_for_status()
    member_info = response.json()
    print(
        f"✅ SUCCESS: Connected as '{member_info.get('displayName')}' (ID: {member_info.get('id')})"
    )
except requests.exceptions.HTTPError as e:
    print(
        f"❌ FAILURE: Could not connect to the API. Status Code: {e.response.status_code}"
    )
    print(f"   Response: {e.response.text}")
except Exception as e:
    print(f"❌ FAILURE: An unexpected error occurred: {e}")

Step 1: Fetching All Dossiers (Records)

In this step, we retrieve a complete list of all dossiers (records) associated with the user account. This list will be the foundation for fetching the meters in the next step. The result is saved to records.csv.

In [ ]:
def fetch_all_records():
    """
    Fetches all records (dossiers) for the user and robustly flattens ALL
    nested data (Structs and Lists of any type) into a CSV-compatible format.
    """
    print("\n--- Starting Step 1: Fetching all records ---")
    records_url = f"{BASE_API_URL}/api/v1/members/me/records"

    try:
        response = requests.get(records_url, headers=headers)
        response.raise_for_status()
        records_data = response.json()

        if not records_data:
            print("No records found for this user.")
            return None

        df_records = pl.from_dicts(records_data)

        # --- FIX 1: Unnest all top-level Struct columns ---
        struct_cols = [
            c for c in df_records.columns if isinstance(df_records[c].dtype, pl.Struct)
        ]
        if struct_cols:
            print(f"   -> Flattening Struct columns: {struct_cols}")
            df_records = df_records.unnest(struct_cols)

        # --- FIX 2: Robustly flatten all List columns ---
        # This new logic handles cases where a list column might be entirely nulls (List[Null]).
        flatten_expressions = []
        for col_name in df_records.columns:
            if isinstance(df_records[col_name].dtype, pl.List):
                # If the list contains complex objects (Structs), serialize the whole list to a JSON string.
                # This is the safest way to preserve all nested data for fields like 'recordUsers'.
                if isinstance(df_records[col_name].dtype.inner, pl.Struct):
                    print(
                        f"   -> Serializing List[Struct] column '{col_name}' to a single JSON string per row."
                    )
                    flatten_expressions.append(
                        pl.col(col_name)
                        .map_elements(
                            lambda lst: json.dumps(lst) if lst else None,
                            return_dtype=pl.String,
                        )
                        .alias(col_name)  # Ensure the original column name is kept
                    )
                # For ALL other list types (String, Null, Int, etc.), cast elements to string and then join.
                # This is the key fix that prevents the "dtype `null`" error.
                else:
                    print(
                        f"   -> Safely joining List column '{col_name}' into a delimited string."
                    )
                    flatten_expressions.append(
                        pl.col(col_name)
                        .list.eval(
                            pl.element().cast(pl.String)
                        )  # Cast every element to string first
                        .list.join("; ")  # Then join
                        .alias(col_name)  # Ensure the original column name is kept
                    )

        # Apply all the list-flattening expressions at once.
        if flatten_expressions:
            df_records = df_records.with_columns(flatten_expressions)

        # Save to CSV
        records_csv_path = EXPORT_DIR / "records.csv"
        df_records.write_csv(records_csv_path)

        print(f"\n✅ Found {len(df_records)} records.")
        print(f"   Saved fully flattened records metadata to '{records_csv_path}'")
        print("\nRecord data preview (after all flattening):")
        # Select first 10 columns for a cleaner preview if the dataframe is very wide
        preview_cols = df_records.columns[:10]
        print(df_records.head(3).select(preview_cols))

        return df_records

    except requests.exceptions.HTTPError as e:
        print(
            f"❌ An HTTP error occurred while fetching records: {e.response.status_code} - {e.response.text}"
        )
        return None
    except Exception as e:
        print(f"❌ An unexpected error occurred while fetching records: {e}")
        # Re-raise to see the full traceback in the notebook for debugging
        raise


# Execute the function
df_records = fetch_all_records()

Step 2: Fetching All Meters

Now, we will iterate through each dossier from the previous step to retrieve all of its associated meters. All meters from all records will be compiled into a single meters.csv file. A small delay is added between each API call to be respectful to the server.

In [ ]:
def fetch_all_meters(records_df: pl.DataFrame):
    """
    Iterates through a DataFrame of records to fetch all associated meters,
    and robustly flattens any nested data before saving.
    """
    if records_df is None or records_df.is_empty():
        print("No records to process, skipping meter fetching.")
        return None

    print("\n--- Starting Step 2: Fetching all meters ---")
    all_meters_list = []
    total_records = len(records_df)

    for i, record in enumerate(records_df.iter_rows(named=True)):
        record_id = record["id"]
        record_name = record["displayName"]

        print(
            f" -> Processing record {i + 1}/{total_records}: '{record_name}' (ID: {record_id})"
        )

        meters_url = f"{BASE_API_URL}/api/v1/records/{record_id}/meters"
        try:
            response = requests.get(meters_url, headers=headers)
            response.raise_for_status()
            meters_data = response.json()

            if meters_data:
                # Add the record_id to each meter dictionary for easier joining later
                for meter in meters_data:
                    meter["recordId_explicit"] = record_id
                all_meters_list.extend(meters_data)

            time.sleep(0.1)  # Small delay to be kind to the API

        except Exception as e:
            print(f"   ❌ Could not fetch meters for record {record_id}: {e}")
            continue

    if not all_meters_list:
        print("\nNo meters found across all records.")
        return None

    df_meters = pl.from_dicts(all_meters_list)

    # --- PROACTIVE FIX: Flatten any nested data in the meters DataFrame ---
    # Unnest Structs
    struct_cols = [
        c for c in df_meters.columns if isinstance(df_meters[c].dtype, pl.Struct)
    ]
    if struct_cols:
        print(f"   -> Flattening Struct columns in meters data: {struct_cols}")
        df_meters = df_meters.unnest(struct_cols)

    # Flatten Lists
    flatten_expressions = []
    for col_name in df_meters.columns:
        if isinstance(df_meters[col_name].dtype, pl.List):
            if isinstance(df_meters[col_name].dtype.inner, pl.Struct):
                print(
                    f"   -> Serializing List[Struct] column '{col_name}' to JSON string."
                )
                flatten_expressions.append(
                    pl.col(col_name)
                    .map_elements(
                        lambda lst: json.dumps(lst) if lst else None,
                        return_dtype=pl.String,
                    )
                    .alias(col_name)
                )
            else:
                print(
                    f"   -> Safely joining List column '{col_name}' into a delimited string."
                )
                flatten_expressions.append(
                    pl.col(col_name)
                    .list.eval(pl.element().cast(pl.String))
                    .list.join("; ")
                    .alias(col_name)
                )

    if flatten_expressions:
        df_meters = df_meters.with_columns(flatten_expressions)

    # --- End of Fix ---

    meters_csv_path = EXPORT_DIR / "meters.csv"
    df_meters.write_csv(meters_csv_path)

    print(f"\n✅ Found a total of {len(df_meters)} meters.")
    print(f"   Saved all flattened meter metadata to '{meters_csv_path}'")
    print("\nMeters data preview:")
    print(df_meters.head(3))

    return df_meters


# Execute the function
if "df_records" in locals():
    df_meters = fetch_all_meters(df_records)

Step 3: Fetching All Readings for Each Meter

This is the most intensive step. We will loop through every meter and download its entire history of readings. The EnergyID API paginates this data, so we will make repeated calls for each meter until all pages have been retrieved. Each meter's data will be saved to its own CSV file to keep file sizes manageable.

Structure: export_directory/readings/{record_id}/{meter_id}.csv

This process can take a significant amount of time depending on the number of meters and the volume of data.

In [ ]:
def fetch_all_readings(meters_df: pl.DataFrame):
    """
    Iterates through meters to fetch and save all their readings.
    This function is robust and includes:
    - Automatic resume: Skips meters that have already been downloaded.
    - Retry logic with exponential backoff for handling temporary API errors (e.g., 429).
    """
    if meters_df is None or meters_df.is_empty():
        print("No meters to process, skipping reading fetching.")
        return

    print("\n--- Starting Step 3: Fetching all readings (this may take a while) ---")
    total_meters = len(meters_df)

    # --- Configuration for Retry Logic ---
    MAX_RETRIES = 5
    INITIAL_BACKOFF = 1  # seconds

    for i, meter in enumerate(meters_df.iter_rows(named=True)):
        meter_id = meter["id"]
        record_id = meter["recordId"]
        meter_name = meter["displayName"]

        print(
            f"\n({i + 1}/{total_meters}) Processing meter: '{meter_name}' (ID: {meter_id})"
        )

        # --- RESUME LOGIC ---
        # Define the output path and check if it already exists.
        record_readings_dir = READINGS_DIR / str(record_id)
        record_readings_dir.mkdir(exist_ok=True)  # Ensure directory exists
        output_path = record_readings_dir / f"{meter_id}.csv"

        if output_path.exists():
            print(f"   -> ✅ SKIPPING: Data already exists at '{output_path}'.")
            continue

        # --- PAGINATION & FETCHING LOGIC ---
        meter_readings = []
        next_row_key = None
        is_complete = False

        while not is_complete:
            params = {"take": 1000}
            if next_row_key:
                params["nextRowKey"] = next_row_key

            # --- RETRY LOOP ---
            for attempt in range(MAX_RETRIES):
                try:
                    readings_url = f"{BASE_API_URL}/api/v1/meters/{meter_id}/readings"
                    response = requests.get(
                        readings_url, headers=headers, params=params, timeout=30
                    )  # Add timeout
                    response.raise_for_status()

                    # If request is successful, process data and break from retry loop
                    page_data = response.json()
                    readings = page_data.get("readings", [])

                    if readings:
                        meter_readings.extend(readings)

                    next_row_key = page_data.get("nextRowKey")
                    if not next_row_key or not readings:
                        is_complete = True  # Mark as done for this meter

                    break  # Successful attempt, exit retry loop

                except requests.exceptions.HTTPError as e:
                    # Handle specific, retryable HTTP status codes
                    if e.response.status_code in [429, 502, 503, 504]:
                        wait_time = INITIAL_BACKOFF * (2**attempt)
                        print(
                            f"   -> ⚠️ WARNING: Received status {e.response.status_code}. Retrying in {wait_time}s... (Attempt {attempt + 1}/{MAX_RETRIES})"
                        )
                        time.sleep(wait_time)
                    else:
                        # For non-retryable errors (e.g., 401, 403, 404), fail immediately for this meter
                        print(
                            f"   -> ❌ ERROR: Non-retryable HTTP error {e.response.status_code} for meter {meter_id}: {e.response.text}"
                        )
                        is_complete = True  # Stop trying for this meter
                        break  # Exit retry loop
                except requests.exceptions.RequestException as e:
                    # Handle network errors (e.g., connection timeout)
                    wait_time = INITIAL_BACKOFF * (2**attempt)
                    print(
                        f"   -> ⚠️ WARNING: Network error ({e}). Retrying in {wait_time}s... (Attempt {attempt + 1}/{MAX_RETRIES})"
                    )
                    time.sleep(wait_time)

            else:  # This 'else' belongs to the for loop, it runs if the loop completes without a 'break'
                print(
                    f"   -> ❌ FAILURE: All {MAX_RETRIES} retry attempts failed for meter {meter_id}. Skipping."
                )
                is_complete = True  # Give up on this meter and move to the next

        # --- SAVE DATA FOR THE CURRENT METER ---
        if meter_readings:
            df_readings = pl.from_dicts(meter_readings)
            df_readings.write_csv(output_path)
            print(
                f"   -> ✅ SUCCESS: Saved {len(df_readings)} readings to '{output_path}'"
            )
        else:
            print(f"   -> No readings found or an error occurred for meter {meter_id}.")

    print("\n--- Finished fetching all readings. ---")


# Execute the function
if "df_meters" in locals():
    fetch_all_readings(df_meters)

## Export Structure and File Format

The data export is organized into a clear, hierarchical structure within the `energyid_export_{timestamp}` directory. All data is provided in flat CSV format to ensure maximum compatibility.

### `records.csv`
This file contains the metadata for every dossier (also known as a "record") associated with the user account. Each row represents one dossier.

-   **`id`**: The unique identifier for the record. This is the primary key you can use to link to meters.
-   **`displayName`**: The human-readable name of the dossier (e.g., "FI - Karekietstraat 1").
-   **`recordType`**: The type of dossier (e.g., `productionUnit`, `household`).
-   **`address_*` columns**: The nested `address` object from the API has been flattened into separate columns like `streetAddress`, `city`, `postalCode`, and `country`.
-   **List-based columns (`tags`, `installations`, etc.)**: Fields that were originally arrays in the API response have been converted into a single string, with elements separated by a semicolon (`;`).
-   **Complex List-based columns (`recordUsers`, etc.)**: Fields that were arrays of complex objects have been serialized into a JSON string to preserve all nested information.

### `meters.csv`
This file contains the metadata for every meter across all dossiers. Each row represents one meter.

-   **`id`**: The unique identifier for the meter. This is the primary key for the readings data.
-   **`recordId`**: The identifier of the record (dossier) this meter belongs to. You can use this to join with `records.csv`.
-   **`displayName`**: The human-readable name of the meter (e.g., "KAR1 - Elektriciteit injectie").
-   **`metric`**: The type of data the meter measures (e.g., `electricityExport`, `solarPhotovoltaicProduction`).
-   **`unit`**: The unit of measurement (e.g., `kilowattHour`).
-   **`recordId_explicit`**: This column was added during the export process to reliably link back to the record ID, even if the original `recordId` field was ambiguous.

### `readings/` Directory
This directory contains the time-series data for every meter. The structure is designed to be logical and prevent overly large files:

-   **`readings/{record_id}/{meter_id}.csv`**: A nested structure where each meter's data is stored in its own CSV file.
    -   The first level of subdirectories is the **record ID**.
    -   The CSV filename is the **meter ID**.

Each readings CSV file has a simple structure:
-   **`timestamp`**: The timestamp for the reading (in ISO 8601 format).
-   **`value`**: The measured value at that timestamp.
-   **Other columns (`min`, `max`, `mean`, `status`, etc.)**: Additional data provided by the API for that reading, if available.

In [ ]:
def create_consolidated_parquet(
    export_dir: Path, records_df: pl.DataFrame, meters_df: pl.DataFrame
):
    """
    Reads all individual CSV files with potentially different schemas and consolidates
    them into a single, joined Parquet file for easier analysis.
    """
    print(
        "\n--- Starting Bonus Step: Consolidating all data into a single Parquet file ---"
    )

    try:
        readings_dir = export_dir / "readings"
        all_paths = list(readings_dir.glob("**/*.csv"))

        if not all_paths:
            print("   -> SKIPPING: No reading files found to consolidate.")
            return

        list_of_lazy_frames = []
        for p in all_paths:
            meter_id = p.stem
            record_id = p.parent.name

            lf = pl.scan_csv(p).with_columns(
                pl.lit(meter_id).alias("meter_id"), pl.lit(record_id).alias("record_id")
            )
            list_of_lazy_frames.append(lf)

        print(
            f"   -> Found {len(list_of_lazy_frames)} reading files. Consolidating schemas..."
        )
        lazy_readings = pl.concat(list_of_lazy_frames, how="diagonal_relaxed")

        lazy_full_df = lazy_readings.join(
            meters_df.lazy(), left_on="meter_id", right_on="id", how="left"
        ).join(
            records_df.lazy(),
            left_on="record_id",
            right_on="id",
            how="left",
            suffix="_record",
        )

        print("   -> Joining all data and preparing to write Parquet file...")

        # --- FIX: Replaced deprecated 'streaming=True' with the modern 'engine="streaming"' ---
        full_df = lazy_full_df.collect(engine="streaming")

        parquet_path = export_dir / "consolidated_data.parquet"
        full_df.write_parquet(parquet_path, compression="zstd")

        print(f"✅ SUCCESS: All data has been consolidated into '{parquet_path}'")
        print(f"   Total rows in consolidated file: {len(full_df)}")

    except Exception as e:
        print(f"❌ FAILURE: Could not create the consolidated Parquet file. Error: {e}")
        raise


# Execute the function
if (
    "df_records" in locals()
    and df_records is not None
    and "df_meters" in locals()
    and df_meters is not None
):
    create_consolidated_parquet(EXPORT_DIR, df_records, df_meters)

In [ ]:
def create_readme_file(export_dir: Path):
    """Creates an informative README.md file inside the export directory."""

    readme_content = f"""
# EnergyID Data Export

**Export Date:** {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

This archive contains a complete data export from an EnergyID user account. The data is provided in two formats: a collection of flat CSV files and a single consolidated Parquet file.

## File Structure

-   `consolidated_data.parquet`: A single file containing all records, meters, and readings joined together. **This is the recommended file for most analyses.**
-   `records.csv`: Metadata for each dossier/record.
-   `meters.csv`: Metadata for every meter across all dossiers.
-   `readings/`: A directory containing the raw time-series data, with one CSV file per meter, organized by `record_id`.
    -   `readings/{{record_id}}/{{meter_id}}.csv`

## How to Load the Data

### Using Polars (Recommended)

```python
import polars as pl

# Load the consolidated Parquet file (fastest and easiest)
df = pl.read_parquet("consolidated_data.parquet")

print(df.head())```

### Using Pandas

```python
import pandas as pd

# Load the consolidated Parquet file
df = pd.read_parquet("consolidated_data.parquet")

print(df.head())
```

## Data Schema Highlights

-   **Joining Keys**:
    -   `records.csv` <> `meters.csv` on `records.id` = `meters.recordId`
    -   `meters.csv` <> Readings via `meters.id` = `readings/.../{{meter_id}}.csv`
-   **Data Flattening**: Nested JSON objects (like addresses) have been flattened into separate columns. Array fields (like tags) have been joined into semicolon-delimited strings.

"""

    try:
        readme_path = export_dir / "README.md"
        with open(readme_path, "w", encoding="utf-8") as f:
            f.write(readme_content)
        print(f"✅ SUCCESS: Created README.md file at '{readme_path}'")
    except Exception as e:
        print(f"❌ FAILURE: Could not create README.md file. Error: {e}")


# Execute the function
create_readme_file(EXPORT_DIR)

In [ ]:
import polars as pl

print("--- Starting General Exploratory Data Analysis of consolidated_data.parquet ---")

# --- Configuration ---
PARQUET_FILE = EXPORT_DIR / "consolidated_data.parquet"

try:
    df = pl.read_parquet(PARQUET_FILE)
    print(f"Successfully loaded {len(df)} rows from the Parquet file.")

    # --- 1. Schema Overview ---
    print("\n--- Schema ---")
    print("The following columns are available in the consolidated dataset:")
    for col_name, dtype in df.schema.items():
        print(f"- {col_name}: {dtype}")

    # --- 2. General Summary Statistics ---
    print("\n--- Summary Statistics ---")
    # .describe() gives a powerful overview of all columns
    print(df.describe())

    # --- 3. Investigate Key Categorical Columns ---
    # This is the most crucial part for finding the group/site identifier.

    print("\n--- Unique Value Analysis for Key Columns ---")

    # Analyze the 'tags' column by splitting the concatenated string back into a list
    if "tags" in df.columns:
        print("\n[Analysis of 'tags' column]")
        unique_tags = (
            df.select(
                pl.col("tags")
                .str.split("; ")  # Split the string we created back into a list
                .list.explode()  # Explode the list to get one tag per row
            )
            .get_column("tags")
            .value_counts()  # Count occurrences of each unique tag
            .sort("count", descending=True)
        )
        print("Most common tags found across all dossiers:")
        print(unique_tags.head(20))  # Print the top 20 most common tags

    # Analyze the 'displayName_record' column
    if "displayName_record" in df.columns:
        print("\n[Analysis of 'displayName_record' column]")
        # It's possible the site is a prefix, like 'WPN - ...' or 'OTB - ...'
        # Let's count the occurrences of the first word/part of the display name.
        name_prefixes = (
            df.select(
                pl.col("displayName_record")
                .str.split(" ")
                .list.first()  # Get the first word
                .alias("name_prefix")
            )
            .get_column("name_prefix")
            .value_counts()
            .sort("count", descending=True)
        )
        print("Most common prefixes found in record display names:")
        print(name_prefixes.head(20))

except FileNotFoundError:
    print(f"❌ ERROR: The consolidated file was not found at '{PARQUET_FILE}'.")
except Exception as e:
    print(f"❌ An unexpected error occurred during analysis: {e}")
    raise

In [ ]:
import polars as pl

print(
    "\n--- Starting final step: Creating client-specific export files for the 'FI' project ---"
)

# --- Configuration ---
PARQUET_FILE = EXPORT_DIR / "consolidated_data.parquet"
PROJECT_PREFIX = "FI - "

INJECTION_CSV_OUTPUT = EXPORT_DIR / "otterbeek_site_injectie_data.csv"
PRODUCTION_CSV_OUTPUT = EXPORT_DIR / "otterbeek_site_productie_data.csv"

try:
    lazy_df = pl.scan_parquet(PARQUET_FILE)

    # Filter for records where 'displayName_record' starts with the project prefix
    print(
        f" -> Filtering the master dataset for records where 'displayName_record' starts with '{PROJECT_PREFIX}'..."
    )
    site_df = lazy_df.filter(
        pl.col("displayName_record").str.starts_with(PROJECT_PREFIX)
    )

    # --- FINAL FIX: Use the correct column 'record_id' which we know exists ---
    # Verify how many unique dossiers were found with this filter
    unique_dossiers_found = (
        site_df.select(pl.col("record_id").n_unique())
        .collect(engine="streaming")
        .item()
    )

    if unique_dossiers_found == 0:
        print(
            f"❌ CRITICAL ERROR: No records found starting with '{PROJECT_PREFIX}'. Halting."
        )
    else:
        print(f"   -> Found {unique_dossiers_found} unique dossiers for the project.")

        # 1. Create the Injection Data CSV
        print(" -> Generating injection data CSV...")
        injection_df = (
            site_df.filter(pl.col("metric") == "electricityExport")
            .select(
                [
                    pl.col("displayName_record").alias("Adres"),
                    pl.col("connectionNumber").alias("EAN"),
                    pl.col("timestamp"),
                    pl.col("value").alias("Injectie (kWh)"),
                ]
            )
            .sort(["Adres", "timestamp"])
            .collect(engine="streaming")
        )

        if not injection_df.is_empty():
            injection_df.write_csv(INJECTION_CSV_OUTPUT)
            print(
                f"   ✅ SUCCESS: Saved {len(injection_df)} injection records to '{INJECTION_CSV_OUTPUT}'"
            )
        else:
            print(
                "   - WARNING: No injection data (metric='electricityExport') found for this project."
            )

        # 2. Create the Production Data CSV
        print(" -> Generating production data CSV...")
        production_df = (
            site_df.filter(pl.col("metric") == "solarPhotovoltaicProduction")
            .select(
                [
                    pl.col("displayName_record").alias("Adres"),
                    pl.col("connectionNumber").alias("EAN"),
                    pl.col("timestamp"),
                    pl.col("value").alias("Productie (kWh)"),
                ]
            )
            .sort(["Adres", "timestamp"])
            .collect(engine="streaming")
        )

        if not production_df.is_empty():
            production_df.write_csv(PRODUCTION_CSV_OUTPUT)
            print(
                f"   ✅ SUCCESS: Saved {len(production_df)} production records to '{PRODUCTION_CSV_OUTPUT}'"
            )
        else:
            print(
                "   - WARNING: No production data (metric='solarPhotovoltaicProduction') found for this project."
            )

        print("\n--- Client-specific export file creation complete. ---")

except FileNotFoundError:
    print(f"❌ ERROR: The consolidated file was not found at '{PARQUET_FILE}'.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")
    raise

Step 4: Zipping the Export

For convenience and portability, the final step is to compress the entire export directory into a single .zip archive.

In [ ]:
def zip_export_directory(export_dir: Path):
    """Zips the entire export directory."""
    print("\n--- Starting Final Step: Archiving the export directory ---")
    zip_path = export_dir.with_suffix(".zip")

    try:
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
            for file in export_dir.glob("**/*"):
                if file.is_file():
                    # Add file to the zip, with a relative path inside the zip
                    arcname = file.relative_to(export_dir)
                    zipf.write(file, arcname)

        print(f"✅ SUCCESS: Export directory successfully zipped to '{zip_path}'")
    except Exception as e:
        print(f"❌ FAILURE: Could not create zip archive. Error: {e}")


# Execute the function
zip_export_directory(EXPORT_DIR)

Export Complete

The process is finished. All user data has been downloaded and archived into a single zip file.

You can now find the file named energyid_export_{timestamp}.zip in the same directory as this notebook. This archive contains all the data in a structured CSV format, ready for analysis.